In [1]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlretrieve

import pyarrow.parquet as pq
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms

torch.set_num_threads(2)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,) * 3, (0.5,) * 3)])

def load_cifar(split, size):
    path = Path(f"cifar10-{split}.parquet")
    if not path.exists():
        url = f"https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/{split}-00000-of-00001.parquet"
        partial = path.with_suffix(".part")
        urlretrieve(url, partial)
        partial.replace(path)
    examples = []
    for batch in pq.ParquetFile(path).iter_batches(batch_size=512):
        for row in batch.to_pylist():
            with Image.open(BytesIO(row["img"]["bytes"])) as image:
                examples.append((transform(image.convert("RGB")), row["label"]))
            if len(examples) == size:
                return examples
    raise ValueError("CIFAR-10 contains fewer examples than requested")

train = load_cifar("train", 6000)
test = load_cifar("test", 1200)
classes = "airplane automobile bird cat deer dog frog horse ship truck".split()

def cnn(filters=24, kernel=3):
    return nn.Sequential(
        nn.Conv2d(3, filters, kernel, padding=kernel // 2), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(filters, filters * 2, kernel, padding=kernel // 2), nn.ReLU(),
        nn.AdaptiveAvgPool2d(4), nn.Flatten(), nn.Linear(filters * 2 * 16, 10)
    ).to(device)

def fit(model, optimizer, epochs=4):
    batches = DataLoader(train, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(7))
    for epoch in range(epochs):
        model.train()
        for x, y in batches:
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(x.to(device)), y.to(device))
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        predictions = torch.cat([model(x.to(device)).argmax(1).cpu() for x, _ in evaluation])
    return predictions

evaluation = DataLoader(test, batch_size=128)
labels = torch.cat([y for _, y in evaluation])

model = cnn()
predictions = fit(model, torch.optim.Adam(model.parameters()))
print(f"CIFAR-10 test accuracy: {(predictions == labels).float().mean():.1%}")
print(f"{'Actual':12} Predicted")
for actual, predicted in zip(labels[:10], predictions[:10]):
    print(f"{classes[actual]:12} {classes[predicted]}")

CIFAR-10 test accuracy: 47.7%
Actual       Predicted
cat          cat
ship         automobile
ship         ship
airplane     ship
frog         frog
frog         frog
automobile   automobile
frog         frog
cat          dog
automobile   automobile
